# Architecture B-PR — Multi-agent Single-model with Prompt Repetition

This notebook runs Architecture **B** (multi-agent, single-model) with **Prompt Repetition** enabled.

**Prompt Repetition**: Based on Leviathan et al., "Prompt Repetition Improves Non-Reasoning LLMs" (arXiv:2512.14982, 2025).
The technique repeats user prompts (`<QUERY>` → `<QUERY>\n\n<QUERY>`) to allow each token to attend to all other tokens.

**Architecture B**: Planner → Router → Developer (S/M/L) → Reviewer → Tester, all using Qwen-7B.

In [1]:
import os
import sys
import subprocess
import pathlib

REPO_URL = "https://github.com/LLM4SE-group-15/ArchitecturesForCodeDevelopmentWithLLMs.git"
REPO_DIR = pathlib.Path("/content/ArchitecturesForCodeDevelopmentWithLLMs")

if not REPO_DIR.exists():
    subprocess.run(["git", "clone", REPO_URL, str(REPO_DIR)], check=True)

os.chdir(REPO_DIR)
subprocess.run([sys.executable, "-m", "pip", "install", "-U", "pip"], check=True)
subprocess.run([sys.executable, "-m", "pip", "install", "-r", "requirements.txt"], check=True)

print(f"Using repo at {REPO_DIR.resolve()}")

Cloning into '/content/ArchitecturesForCodeDevelopmentWithLLMs'...


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.8/1.8 MB 24.1 MB/s eta 0:00:00
  Attempting uninstall: pip
    Found existing installation: pip 24.1.2
    Uninstalling pip-24.1.2:
      Successfully uninstalled pip-24.1.2


ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
fastai 2.8.4 requires fastcore<1.9,>=1.8.0, but you have fastcore 1.11.3 which is incompatible.


INFO: pip is looking at multiple versions of langchain-huggingface to determine which version is compatible with other requirements. This could take a while.
INFO: pip is looking at multiple versions of langgraph-prebuilt to determine which version is compatible with other requirements. This could take a while.
INFO: pip is still looking at multiple versions of langgraph-prebuilt to determine which version is compatible with other requirements. This could take a while.
INFO: This is taking longer than usual. You might need to provide the dependency resolver with stricter constraints to reduce runtime. See https://pip.pypa.io/warnings/backtracking for guidance. If you want to abort this run, press Ctrl + C.
INFO: pip is looking at multiple versions of multiprocess to determine which version is compatible with other requirements. This could take a while.
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 527.3/527.3 kB 11.4 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 536.4/536.4 k

ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-adk 1.22.1 requires google-cloud-bigquery-storage>=2.0.0, which is not installed.
bigframes 2.26.0 requires google-cloud-bigquery-storage<3.0.0,>=2.30.0, which is not installed.
google-colab 1.0.0 requires google-auth==2.38.0, but you have google-auth 2.47.0 which is incompatible.
google-colab 1.0.0 requires jupyter-server==2.14.0, but you have jupyter-server 2.12.5 which is incompatible.
google-colab 1.0.0 requires requests==2.32.4, but you have requests 2.32.5 which is incompatible.
cudf-cu12 25.6.0 requires pyarrow<20.0.0a0,>=14.0.0; platform_machine == "x86_64", but you have pyarrow 22.0.0 which is incompatible.
gradio 5.49.1 requires pydantic<2.12,>=2.0, but you have pydantic 2.12.5 which is incompatible.
bigframes 2.26.0 requires rich<14,>=12.4.4, but you have rich 14.2.0 which is incompatible.
fastai

In [2]:
!pip install -r requirements.txt

In [ ]:
import os
import getpass
from huggingface_hub import login

# Configurazione LangSmith
#placeholder

Note: Environment variable`HF_TOKEN` is set and is the current active token independently from the token you've just configured.


ARCHITECTURE set to B
PROMPT_REPETITION enabled


In [4]:
from huggingface_hub import HfApi

api = HfApi()
try:
    user_info = api.whoami(token=os.environ["HF_TOKEN"])
    print("Logged in to Hugging Face as:", user_info.get("name") or user_info.get("user"))
except Exception as exc:
    print("Login check failed:", exc)

Logged in to Hugging Face as: Riaburger


In [5]:
import json
import time
import logging
import pathlib
import os

from datetime import datetime

DEFAULT_ROOT = pathlib.Path("/content/ArchitecturesForCodeDevelopmentWithLLMs")
ROOT = DEFAULT_ROOT if DEFAULT_ROOT.exists() else pathlib.Path.cwd()

sys.path.insert(0, str(ROOT))

LOG_DIR = ROOT / "log"
LOG_DIR.mkdir(exist_ok=True)

logger = logging.getLogger("architecture_B_PR")
logger.setLevel(logging.INFO)
if logger.handlers:
    logger.handlers.clear()
formatter = logging.Formatter("%(asctime)s | %(levelname)s | %(message)s")
file_handler = logging.FileHandler(LOG_DIR / "architecture_B_PR.log")
stream_handler = logging.StreamHandler()
for handler in (file_handler, stream_handler):
    handler.setFormatter(formatter)
    logger.addHandler(handler)

logger.info("Logger ready. Repo root: %s", ROOT)
logger.info("Log files: %s", LOG_DIR)
logger.info("PROMPT_REPETITION: %s", os.environ.get("PROMPT_REPETITION", "false"))
print("Logs ->", LOG_DIR)

2026-01-25 23:37:40,626 | INFO | Logger ready. Repo root: /content/ArchitecturesForCodeDevelopmentWithLLMs
2026-01-25 23:37:40,627 | INFO | Log files: /content/ArchitecturesForCodeDevelopmentWithLLMs/log
2026-01-25 23:37:40,627 | INFO | PROMPT_REPETITION: true


Logs -> /content/ArchitecturesForCodeDevelopmentWithLLMs/log


In [6]:
import time
import json
import random
from src.data.task_loader import HumanEvalTaskLoader
from src.graph.graph import run_graph
from src.agents.llm import Architecture

ARCH = Architecture.B
SEED_FIXED = 31

def run_humaneval_benchmark(limit: int = 15, shuffle: bool = True):
    """
    Run benchmark on HumanEval tasks with Prompt Repetition enabled.
    
    Args:
        limit: Number of tasks to run
        shuffle: If True, randomly sample tasks with fixed seed
    """
    loader = HumanEvalTaskLoader()
    all_tasks = loader.load_all()
    
    if shuffle:
        random.seed(SEED_FIXED)
        tasks = random.sample(all_tasks, min(limit, len(all_tasks)))
    else:
        tasks = all_tasks[:limit]

    results = []
    total = len(tasks)
    logger.info("Loaded %s tasks from HumanEval (shuffle=%s, seed=%s)", total, shuffle, SEED_FIXED)
    logger.info("Prompt Repetition: ENABLED")
    print(f"Starting benchmark on {total} tasks (Prompt Repetition: ON)...")
    
    for idx, task in enumerate(tasks, 1):
        logger.info("Running %s/%s %s", idx, total, task.task_id)
        print(f"[{idx}/{total}] Task {task.task_id} ({task.entry_point})... ", end="", flush=True)
        
        start = time.time()
        
        state = run_graph(
            task_id=task.task_id,
            task_description=task.prompt,
            test_code=task.test,
            entry_point=task.entry_point,
            architecture=ARCH,
        )

        elapsed = time.time() - start
        
        record = {
            "task_id": task.task_id,
            "entry_point": task.entry_point,
            "architecture": "B-PR",
            "prompt_repetition": True,
            "test_passed": state["test_passed"],
            "developer_tier": state.get("developer_tier"),
            "escalations": state["escalations"],
            "story_points_initial": state.get("story_points_initial"),
            "story_points_final": state.get("story_points_current"),
            "elapsed_seconds": elapsed,
        }
        results.append(record)
        
        logger.info(
            "Finished %s | pass=%s tier=%s escalations=%s elapsed=%.1fs",
            task.task_id,
            state["test_passed"],
            record["developer_tier"],
            record["escalations"],
            elapsed,
        )
        status_str = "PASS" if state["test_passed"] else "FAIL"
        print(f"{status_str} in {elapsed:.1f}s")

        with open(LOG_DIR / "architecture_B_PR.jsonl", "a", encoding="utf-8") as f:
            f.write(json.dumps(record) + "\n")
            
    return results

# Esecuzione: 15 task random (seed=31 per riproducibilita)
sample_results = run_humaneval_benchmark(limit=15, shuffle=True)

# Sommario
passed_count = sum(1 for r in sample_results if r['test_passed'])
print(f"\nBenchmark Completed. Passed: {passed_count}/{len(sample_results)}")

Loading HumanEval dataset...


Generating test split:   0%|          | 0/164 [00:00<?, ? examples/s]

2026-01-25 23:37:46,770 | INFO | Loaded 15 tasks from HumanEval (shuffle=True, seed=31)
2026-01-25 23:37:46,770 | INFO | Prompt Repetition: ENABLED
2026-01-25 23:37:46,771 | INFO | Running 1/15 HumanEval/3


Loaded 164 tasks.
Starting benchmark on 15 tasks (Prompt Repetition: ON)...
[1/15] Task HumanEval/3 (below_zero)... 

2026-01-25 23:38:04,143 | INFO | Finished HumanEval/3 | pass=True tier=L escalations=2 elapsed=17.4s
2026-01-25 23:38:04,144 | INFO | Running 2/15 HumanEval/120


...

PASS in 9.8s
[14/15] Task HumanEval/114 (minSubArraySum)... 

2026-01-25 23:41:06,778 | INFO | Finished HumanEval/114 | pass=True tier=M escalations=0 elapsed=8.8s
2026-01-25 23:41:06,780 | INFO | Running 15/15 HumanEval/134


PASS in 8.8s
[15/15] Task HumanEval/134 (check_if_last_char_is_a_letter)... 

2026-01-25 23:41:22,278 | INFO | Finished HumanEval/134 | pass=True tier=M escalations=1 elapsed=15.5s


PASS in 15.5s

Benchmark Completed. Passed: 10/15


In [7]:
!cd log && cat architecture_B_PR.jsonl

{"task_id": "HumanEval/3", "entry_point": "below_zero", "architecture": "B-PR", "prompt_repetition": true, "test_passed": true, "developer_tier": "L", "escalations": 2, "story_points_initial": 2, "story_points_final": 8, "elapsed_seconds": 17.370441675186157}
{"task_id": "HumanEval/120", "entry_point": "maximum", "architecture": "B-PR", "prompt_repetition": true, "test_passed": true, "developer_tier": "M", "escalations": 0, "story_points_initial": 3, "story_points_final": 3, "elapsed_seconds": 9.469397068023682}
{"task_id": "HumanEval/28", "entry_point": "concatenate", "architecture": "B-PR", "prompt_repetition": true, "test_passed": false, "developer_tier": "L", "escalations": 2, "story_points_initial": 1, "story_points_final": 8, "elapsed_seconds": 14.667582273483276}
{"task_id": "HumanEval/100", "entry_point": "make_a_pile", "architecture": "B-PR", "prompt_repetition": true, "test_passed": true, "developer_tier": "S", "escalations": 0, "story_points_initial": 2, "story_points_final"

## Evaluation Metrics for Architecture B-PR

This section calculates the evaluation metrics as specified in `evaluation.md`:

- **Primary Metrics**: Pass Rate, Pass@1
- **Cost Metrics**: Execution Time, API Calls, Escalations
- **Comparison**: B vs B-PR (RQ4 - Prompt Repetition effect)

In [8]:
import json
import pandas as pd

# Load results
results_file = LOG_DIR / "architecture_B_PR.jsonl"
records = []
with open(results_file, "r", encoding="utf-8") as f:
    for line in f:
        if line.strip():
            records.append(json.loads(line))

df = pd.DataFrame(records)
print(f"Loaded {len(df)} task results")
df

Loaded 15 task results


,task_id,entry_point,architecture,prompt_repetition,test_passed,developer_tier,escalations,story_points_initial,story_points_final,elapsed_seconds
0,HumanEval/3,below_zero,B-PR,True,True,L,2,2,8,17.370442
1,HumanEval/120,maximum,B-PR,True,True,M,0,3,3,9.469397
2,HumanEval/28,concatenate,B-PR,True,False,L,2,1,8,14.667582
3,HumanEval/100,make_a_pile,B-PR,True,True,S,0,2,2,11.109190
4,HumanEval/36,fizz_buzz,B-PR,True,True,M,1,2,3,14.039977
5,HumanEval/11,string_xor,B-PR,True,True,S,0,1,1,10.614819
6,HumanEval/35,max_element,B-PR,True,True,S,0,1,1,6.645681
7,HumanEval/137,compare_one,B-PR,True,True,M,0,5,5,11.546548
8,HumanEval/59,largest_prime_factor,B-PR,True,False,L,1,3,8,19.953085
9,HumanEval/37,sort_even,B-PR,True,False,L,2,2,8,28.942281


In [9]:
# Calculate metrics
total_tasks = len(df)
passed_tasks = df['test_passed'].sum()
pass_rate = passed_tasks / total_tasks * 100
avg_time = df['elapsed_seconds'].mean()
total_time = df['elapsed_seconds'].sum()
avg_escalations = df['escalations'].mean()

print("=" * 50)
print("ARCHITECTURE B-PR (Multi-agent + Prompt Repetition)")
print("=" * 50)
print(f"Total Tasks:     {total_tasks}")
print(f"Passed:          {passed_tasks}")
print(f"Pass Rate:       {pass_rate:.1f}%")
print(f"Avg Time/Task:   {avg_time:.2f}s")
print(f"Total Time:      {total_time:.1f}s")
print(f"Avg Escalations: {avg_escalations:.2f}")
print("=" * 50)
print("\nPrompt Repetition: ENABLED")

ARCHITECTURE B-PR (Multi-agent + Prompt Repetition)
Total Tasks:     15
Passed:          10
Pass Rate:       66.7%
Avg Time/Task:   14.36s
Total Time:      215.5s
Avg Escalations: 0.93

Prompt Repetition: ENABLED


In [10]:
# Tier distribution
print("\nDeveloper Tier Distribution:")
print(df['developer_tier'].value_counts())


Developer Tier Distribution:
developer_tier
L    6
M    6
S    3
Name: count, dtype: int64


In [11]:
# Verifica prompt repetition
from src.agents.client import get_llm_client
from src.agents.llm import get_prompt_repetition

print(f"PROMPT_REPETITION env: {os.environ.get('PROMPT_REPETITION')}")
print(f"get_prompt_repetition(): {get_prompt_repetition()}")

client = get_llm_client()
print(f"client.prompt_repetition: {client.prompt_repetition}")

# Test ripetizione
test_messages = [{"role": "user", "content": "Hello world"}]
repeated = client._apply_prompt_repetition(test_messages)
print(f"\nOriginal: {test_messages[0]['content']}")
print(f"Repeated: {repeated[0]['content']}")

PROMPT_REPETITION env: true
get_prompt_repetition(): True
client.prompt_repetition: True

Original: Hello world
Repeated: Hello world

Hello world
